# 5.  Execution and Visualisation
The main execution script orchestrates the full pipeline

In [ ]:
def run_full_analysis(depths=[2, 3, 5], device='cpu'):
    """Execute complete experimental pipeline.

    Args:
        depths: List of convolutional layer counts
        device: 'cpu' or 'cuda'

    Returns:
        results: Nested dict with all measurements
    """
    results = {
        'probe': {},
        'rsa': {},
        'tcav': {},
        'intervention': {}
    }

    # Load data once
    train_loader, test_loader = get_mnist_loaders(batch_size=128)

    for depth in depths:
        print(f'\n[Training] Depth={depth}')

        # Initialise and train model
        model = MNIST_CNN(depth=depth)
        model = train_model(model, train_loader, test_loader,
                           device=device)

        # Save checkpoint
        torch.save(model.state_dict(),
                   f'./models/mnist_depth{depth}.pt')

        layer_indices = list(range(depth))

        for concept, cfg in CONCEPT_MAP.items():
            print(f'  [Analysing] {concept}')

            # Create concept subsets
            pos_ds, neg_ds = get_concept_subsets(
                train_loader.dataset, concept)
            pos_loader = DataLoader(pos_ds, batch_size=64)
            neg_loader = DataLoader(neg_ds, batch_size=64)

            # Extract activations at all layers
            pos_acts_all = []
            neg_acts_all = []
            for li in layer_indices:
                pos_acts = extract_activations(model, pos_loader, li, device)
                neg_acts = extract_activations(model, neg_loader, li, device)
                pos_acts_all.append(pos_acts)
                neg_acts_all.append(neg_acts)

            # 1. CAV & Probing (per layer)
            for li in layer_indices:
                cav = compute_cav(pos_acts_all[li], neg_acts_all[li])

                acts_combined = np.vstack([
                    pos_acts_all[li],
                    neg_acts_all[li]
                ])
                labels_combined = np.concatenate([
                    np.ones(len(pos_acts_all[li])),
                    np.zeros(len(neg_acts_all[li]))
                ])

                probe_mean, probe_std = linear_probe(
                    acts_combined, labels_combined)

                key = f'depth{depth}_layer{li}_{concept}'
                results['probe'][key] = {
                    'mean': probe_mean,
                    'std': probe_std
                }

            # 2. RSA (per layer)
            rsa_sample = extract_activations(
                model, test_loader, 0, device,
                max_samples=500, stratified=True)

            for li in layer_indices:
                acts = extract_activations(
                    model, test_loader, li, device,
                    max_samples=500, stratified=True)

                rdm_net = compute_rdm(acts)
                rdm_concept = make_concept_rdm(
                    concept_labels_for_sample)

                tau, pval = rsa_correlation(rdm_net, rdm_concept)
                tau_obs, ci_lo, ci_hi = rsa_bootstrap_ci(
                    rdm_net, rdm_concept)

                key = f'depth{depth}_layer{li}_{concept}'
                results['rsa'][key] = {
                    'tau': tau, 'pvalue': pval,
                    'ci_lower': ci_lo, 'ci_upper': ci_hi
                }

            # 3. Intervention (final layer only)
            final_layer = layer_indices[-1]
            cav = compute_cav(
                pos_acts_all[final_layer],
                neg_acts_all[final_layer])

            # Ablation
            abl_before, abl_after, abl_delta = targeted_ablation(
                model, test_images, test_labels,
                concept_mask, final_layer)

            # Injection
            inj_mean, inj_std = counterfactual_injection(
                model, test_images, test_labels,
                cav, final_layer, alpha=1.0)

            key = f'depth{depth}_{concept}'
            results['intervention'][key] = {
                'ablation_delta': abl_delta,
                'injection_mean': inj_mean,
                'injection_std': inj_std
            }

    return results
